<div style="font-family: Arial, sans-serif; line-height: 1.6; color: #fab956; max-width: 1200px; margin: 0 auto; padding: 20px; border: 1px solid #000000; border-radius: 8px; background-color: #00183a74;">
  <h1 style="color: #c25719; border-bottom: 2px solid #c25719; padding-bottom: 8px; margin-top: 0;">
    Notebook Map
  </h1>

In this notebook we provide a summary of the work carried out. On the following notebooks one is able to find the whole process, decision making, tests and analysis.
  
| # | Notebook | Content | CRISP-DM phase |
|---|---|---|---|
| 00 | `00_relatorio.ipynb` | Problem, data, literature, decisions | Business and data understanding |
| 01 | `01_lndb_exploration.ipynb` | LNDb exploration and the LNDb nodule table | Data understanding |
| 02 | `02_lidc_exploration.ipynb` | LIDC IDRI exploration and the LIDC nodule table | Data understanding |
| 03 | `03_tabela_nodulos.ipynb` | Joined table, study cohort, labels (median of the readers) on both scales | Data preparation |
| 04 | `04_patches.ipynb` | 3D patches and their checks | Data preparation |
| 05 | `05_features.ipynb` | Radiologists' regions, PyRadiomics (original, LoG and wavelet images), robustness, feature filters | Data preparation |
| 06 | `06_modelo.ipynb` | Features, ways of pooling the datasets, models, binary answer, nested validation, validation across datasets, interpretability | Modelling and evaluation |

Shared code: `src/mass_detect/scripts/` (`config`, `schema`, `lidc`, `cohort`, `patches`, `qc`, `rois`, `radiomics_features`, `feature_tables`, `sensitivity`, `models`, `modelling`, `run_models`). Unit tests: `src/mass_detect/tests` (run at the start of notebooks 04 and 06).

  <h1 style="color: #c25719; border-bottom: 2px solid #c25719; padding-bottom: 8px; margin-top: 0;">
    Predicting Lung Nodule Malignancy from CT Data
  </h1>

  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
   The problem and who uses it
  </h3>
  <p>
  The end user is a thoracic radiologist who reads a chest CT with lung nodules and must decide what to do with each one: nothing, a follow up scan, or further investigation. Our model gives a second opinion on the malignancy rating of a nodule that the radiologist has already found and outlined, on the original 1 to 5 scale, and as a binary answer that follows that clinical decision: suspicious (a rating of 4 or 5), which calls for further investigation, or not.
  </p>
  <p>
  Constraints we work under:
  </p>
  <ul style="padding-left: 20px; margin-bottom: 20px;">
    <li style="margin-bottom: 8px;">The label is a subjective rating, and two radiologists agree only moderately, which sets a ceiling for any model.</li>
    <li style="margin-bottom: 8px;">The model must be explainable, because a radiologist will not act on a score they cannot trully understand.</li>

  </ul>
  <p>
  This report uses training data only.
  </p>

  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
   What data do we have?
  </h3>

  <p>
  
| Dataset | What it contains | How we use it |
|---|---|---|
| LNDb train | 236 CT scans, with 1219 unique findings (768 nodules), segmentation masks for nodules >= 3 mm | Essential training data |
| LIDC IDRI | 1010 patients, 1018 CT series, up to four readings per nodule | Annotations through `pylidc`; patches cut from the DICOM files; pooled with LNDb in training and validation |
  </p>
  <p>
  Malignancy is evaluated in LNDb's file <strong>'chars_trainNodules.csv'</strong>, joined to the readings by row position, because RadID means different things in the two files.
  </p>
  
  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
    Which files to consider? And which to discard?
  </h3>

  <ol style="padding-left: 20px; margin-bottom: 20px;">
    <li style="margin-bottom: 8px;">Kept:
      <ul style="padding-left: 20px; margin-bottom: 20px;">
      <li style="margin-bottom: 8px;">The merged findings file <strong>'trainNodules_gt.csv'</strong>. The unit of analysis is the nodule, not the reading, because one nodule can be read by up to three radiologists and counting it three times would leak information which is a crucial error we made sure not to make.</li>
      <li style="margin-bottom: 8px;"><strong>'chars_trainNodules.csv'</strong> for the malignancy ratings, <strong>'LNDbAcqParams.csv'</strong> for scanner information, and the masks for nodules of 3 mm or more.</li>
      </ul>
    </li>
    <li style="margin-bottom: 8px;">Dropped:
      <ul style="padding-left: 20px; margin-bottom: 20px;">
        <li style="margin-bottom: 8px;">The per patient diagnosis file of LIDC. It says whether a patient has cancer but not which nodule is the malignant one, so using it as a label would give every nodule of that patient the same wrong answer. Therefore, we did not consider it useful for our main goal.</li>
        <li style="margin-bottom: 8px;">Fleischner files and medical reports. Fleischner is a follow up recommendation for the patient, not a malignancy rating for each nodule, and the reports are free text, not a rating.</li>
        <li style="margin-bottom: 8px;">The original <strong>'.rar'</strong> archives, the X ray series of LIDC (DX and CR) and the macOS <strong>'._*'</strong> files.</li>
        </ul>
    </li>
    <li style="margin-bottom: 8px;">Practical Notes:</li>
      <ul style="padding-left: 20px; margin-bottom: 20px;">
        <li style="margin-bottom: 8px;"><strong>'allNods.csv'</strong> holds ratings already averaged, but its findings were merged differently (the same FindingID points to another place in 90 findings), so we ended up not considering it.</li>
      </ul>
    </li>
  </ol>

  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
   Key Findings in the Literature
  </h3>

  <p>
  <strong>Size alone explains a lot:</strong> Baltatzis et al. (2021) show that a baseline that only uses nodule diameter reaches 83% to 85% accuracy on LIDC, and that the way labels are aggregated can matter more than the model. Their task is binary and drops the nodules rated 3, so these figures are not comparable with ours, which keep every nodule. The same holds for the high accuracies of deep learning studies on LIDC IDRI, such as Causey et al. (2018), which also separate benign from malignant nodules. With this in mind we will always report a diameter only baseline and use it for every comparison. Our diameter only baseline uses the diameter measured on the radiologists' region.
  </p>
  <p>
  <strong>Splits must respect patients:</strong> nodules from one scan share scanner, protocol and anatomy, so placing nodules of the same patient on both sides of a split leaks information (Kapoor and Narayanan, 2023), and we always validate by patient.
  </p>
  <p>
  <strong>Tools:</strong> PyRadiomics is our feature extractor and the IBSI work (Zwanenburg et al., 2020) is the reference for feature definitions. The IBSI requires isotropic voxels for 3D texture features but does not prescribe a size specifically. Taking this into consideration we decided to test diferent values and end up using 0.6 mm, chosen with a sensitivity analysis on our data (notebook 05): it predicted better than 1 mm, kept every small nodule and gave the features most stable between readers, at the cost of a stronger dependence on the slice thickness of the scanner. The bin width is fixed at 25 HU.
  </p>
  <p>
  <strong>Domain shift:</strong> LNDb comes from one hospital with three Siemens scanner models, while LIDC is multi centre and multi vendor. LNDb was designed as an external set for models trained on LIDC. Both datasets enter training and validation. Notebook 06 compares ways of combining them and it also trains on one dataset and validates on the other, which measures how the model would behave in a new hospital. This is also one of our priorities since we have the goal of developing a good generalization model.
  </p>
  <p>


  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
   How the label is built
  </h3>
  <ul style="padding-left: 20px; margin-bottom: 20px;">
    <li style="margin-bottom: 8px;"><strong>The unit is the nodule, not the reading.</strong> Counting readings would weigh a nodule by the number of its readers and could leak the same nodule across folds.</li>
    <li style="margin-bottom: 8px;"><strong>The median of the readers.</strong> A single reader who disagrees with the others cannot move the median, while it moves the mean. Using the mean, for instance, will also increase the number of nodules classified as 3 not because they are the most common but because of the measure itself. The median is also defined for every nodule, while the most frequent rating (the mode) is tied whenever two readers disagree, and with up to three readers the mode, with ties broken by the median, is the median itself. With this in mind we decided to use the median while further analysis on this is present on the other notebooks. The mean and the mode are kept only to compare with the median in notebook 06.</li>
    <li style="margin-bottom: 8px;"><strong>Rounded half up, floor(median + 0.5).</strong> With an even number of readers the median can end in .5. This decision of rounding half up is related to our business understanding and we considered the most caution rating since we are working with a clinical goal. </li>
    <li style="margin-bottom: 8px;"><strong>The binary scale comes from the rounded rating.</strong> A nodule is suspicious when its rating is 4 or 5: in the rating scale of both datasets 3 means indeterminate and only 4 and 5 express a suspicion of cancer. Notebook 03 gives the full justification.</li>
    <li style="margin-bottom: 8px;"><strong>Class 3 is kept.</strong> It is 33% of the LNDb cohort and 46% of the LIDC IDRI cohort, and on the binary scale it counts as not suspicious. Removing it makes the problem look easier than it is (Lei et al., 2020; Zhang et al., 2022; Baltatzis et al., 2021).</li>
    <li style="margin-bottom: 8px;"><strong>The label is noisy.</strong> In LNDb, 20% of the nodules read by several radiologists differ by 2 points or more; in LIDC IDRI 40% do, and 23% were called benign by one reader and suspicious by another.</li>
  </ul>

  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
   Decisions
  </h3>

| Decision | Choice | Why |
|----|----|----|
| Scope | Training data only | The test set has no labels |
| Unit of analysis | Unique nodule | Section above |
| Cohort | Nodule with a rating; 3 mm or more with a radiologist mask; LIDC groups of at most four annotations with a confirmed position. 622 LNDb + 2615 LIDC = 3237 nodules from 1062 patients | The same rule in both datasets |
| Label | floor(median + 0.5) of the non zero ratings | Section above. |
| Scales | 1 to 5, and binary: suspicious for a rating of 4 or 5 (both from the rounded rating). Both are final answers, given by the same model | The task, and the clinical decision to investigate a nodule (McNitt-Gray et al., 2007; MacMahon et al., 2017) |
| Region of interest | Consensus of the radiologists' masks (a voxel is kept when at least half of the readers drew it), plus each reader's mask | The pylidc rule; Ross et al. (2007); Carrazza et al. (2016) |
| Patch | 64 mm cube | 0 consensus regions cut at 64 mm, against 117 at 30 mm and 5 at 51 mm; 99.6% of the nodules keep a 9 mm margin (3σ of the widest LoG filter) |
| Voxel | Isotropic 0.6 mm | The IBSI asks for isotropic voxels without fixing their size. In a sensitivity analysis at 0.6, 1, 1.5 and 2 mm, 0.6 mm predicted best (mean QWK +0.022 over 1 mm, 95% interval +0.009 to +0.036), kept every small nodule and gave the highest ICC (notebook 05) |
| Features | PyRadiomics on the consensus region (original image, LoG 1, 2 and 3 mm and the eight wavelet images, bin width 25 HU). Four filters that never use the label: more than 20% missing, constant, ICC(2,1) below 0.75 in either dataset, Spearman above 0.9 with another feature. 63 of the 386 features of the original and LoG images are kept, and 169 of 1130 with the wavelet images. The model uses the 63 | Demircioğlu (2021); Traverso et al. (2018); Shrout and Fleiss (1979); Koo and Li (2016). The wavelet features did not improve the mean QWK beyond the noise (0.010, interval from 0.002 lower to 0.023 higher) and transfer worse from one dataset to the other |
| Diameter | Measured on the consensus region | One definition for both datasets (Bankier et al., 2017) |
| Pooling LNDb and LIDC | One model for both datasets, compared with a dataset indicator and ComBat with LIDC IDRI as the reference batch | None beat plain pooling beyond the noise |
| Models | Ordinal logistic regression with an L2 penalty (McCullagh, 1980) | Compared with three different non linear families, chosen because they are the usual strong options for radiomic and tabular data (Parmar et al., 2015; Grinsztajn et al., 2022) and are used in reference works of the brief (Shen et al., 2015; Liu et al., 2018; Causey et al., 2018): random forest, gradient boosting and an RBF SVM, wrapped with the ordinal method of Frank and Hall (2001) and tuned as carefully. None beat it beyond the noise. The diameter alone reaches a mean QWK of 0.218 but the same AUC as the full model, 0.860 |
| Extensions of the chosen model | Rare ratings weighted up (weight proportional to one over the share of the rating, power 0.5, chosen in the inner folds) | The only extension above the unweighted model beyond the noise: 0.016 more (interval from 0.002 to 0.032). |
| Answer of the model | The most probable rating on the 1 to 5 scale; suspicious when the probability of a rating of 4 or 5 passes the threshold with the best balanced accuracy, chosen in the inner folds | Hastie et al. (2009); Gneiting (2011); Youden (1950); Fluss et al. (2005); Perkins and Schisterman (2006); Brodersen et al. (2010); Leeflang et al. (2008). |
| Validation | Nested StratifiedGroupKFold by patient (5 outer folds repeated with 3 seeds; 3 inner folds drawn with 2 seeds); intervals by resampling patients; an option replaces the simplest one only if the interval of its difference lies above zero; the final choice is confirmed on 3 new seeds | Cawley and Talbot (2010); Varoquaux (2018); Field and Welsh (2007); Kapoor and Narayanan (2023); Krstajic et al. (2014) |
| Generalisation | Trained on one dataset and validated on the other, compared on the same nodules with cross validation inside that dataset | Steyerberg and Harrell (2016); Zech et al. (2018) |
| Hyperparameters | Chosen in the inner folds with the one standard error rule: the simplest candidate within one standard error of the best. Ordinal penalty: ten values from 0.1 to 3000 on a logarithmic scale. Random forest and gradient boosting: random search of 15 and 20 candidates; SVM: 56 pairs of C and gamma. Grid check fixed in advance: no end of a grid chosen in more than 20% of the outer folds | Hastie et al. (2009); Bergstra and Bengio (2012); Probst, Wright and Boulesteix (2019); Probst and Boulesteix (2018); Hsu, Chang and Lin (2003). |
| Metrics | Primary, fixed in advance: the mean of the QWK in LNDb and in LIDC IDRI (1 to 5) and the mean of the AUC (binary). Secondary: balanced accuracy, MAE, sensitivity, specificity, calibration and the agreement with every single radiologist.  |
| Interpretability | Permutation importance | Molnar (2022) |
<p>
</p>
  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
   Main results (notebook 06)
  </h3>
  <ul style="padding-left: 20px; margin-bottom: 20px;">
    <li style="margin-bottom: 8px;"><strong>Chosen configuration:</strong> ordinal logistic regression on the 63 robust features of the original and LoG images, with the rare ratings weighted up, one model for both datasets, trained on the median rating, giving the most probable rating and a binary answer with a threshold chosen in the inner folds.</li>
    <li style="margin-bottom: 8px;"><strong>Performance on three new seeds:</strong> mean QWK 0.621 (95% interval from 0.576 to 0.661) on the 1 to 5 scale, 0.557 in LNDb and 0.686 in LIDC IDRI; mean AUC 0.861 (from 0.833 to 0.886) on the binary scale, with a sensitivity of 0.50 and a specificity of 0.90 in LNDb, and 0.82 and 0.85 in LIDC IDRI. The seeds used for choosing gave 0.624, so the choices made the estimate only about 0.003 optimistic.</li>
    <li style="margin-bottom: 8px;"><strong>A new hospital:</strong> trained on LIDC IDRI only, the model keeps its performance in LNDb (QWK 0.541 against 0.546 inside LNDb); trained on LNDb only, it loses 0.040 in LIDC IDRI. The ranking of the nodules transfers, but the binary threshold does not: it must be set again on local data. The wavelet features transfer worse.</li>
    <li style="margin-bottom: 8px;"><strong>Median, mean or mode:</strong> the three rules give the same agreement with every single radiologist; they only differ for the nodules read by three or four radiologists, where the median is the easiest label to learn.</li>
    <li style="margin-bottom: 8px;"><strong>What did not help beyond the noise:</strong> the wavelet features, a dataset indicator, ComBat, random forest, gradient boosting, the SVM and the elastic net.</li>
    <li style="margin-bottom: 8px;"><strong>What the model looks at:</strong> size (the two shortest axes) and density (mean and brightest value), then texture heterogeneity, the same signs radiologists use.</li>
    <li style="margin-bottom: 8px;"><strong>Main weaknesses:</strong> the model is still drawn towards rating 3 for many nodules rated 2 or 4; one threshold for both datasets misses half of the suspicious LNDb nodules; and its probabilities are too high, so they need a recalibration before being read as risks.</li>
  </ul>
    <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
    Model interpretability and fairness
  </h3>

  <p>
    Beyond accuracy, a model that supports a clinical decision must answer why it gives a score (interpretability) and whether it gives it fairly to the groups it will meet in practice (fairness). Both are examined in notebook 06. 
  </p>

  <p>
    <strong>Interpretability.</strong>
    The chosen model is an ordinal logistic regression, interpretable by construction: each of the 63 robust features carries one coefficient per cumulative decision. Two extra readings are reported, because coefficients on standardised features are not on the scale a radiologist uses and correlated features can hide each other's contribution.
  </p>
  <ul style="padding-left: 20px; margin-bottom: 20px;">
    <li style="margin-bottom: 8px;"><strong>Permutation importance</strong> (Molnar, 2022): this asks which inputs the model actually relies on when it rates a nodule it has never seen. Each feature is shuffled inside the held out fold, which breaks its link with the rating while leaving the other features intact, and the drop of the mean QWK measures how much the model needed it. The answer is short: three features carry most of the model, the two shortest axes of the consensus region and its mean density, and no other feature lowers the QWK by more than 0.01. A rating can therefore be explained in the terms a radiologist already uses.</li>
    <li style="margin-bottom: 8px;"><strong>Agreement with the radiologists' signs</strong>: the features that matter most in the model coincide with the ones the LIDC protocol asks radiologists to judge.</li>
  </ul>

  <p>
    <strong>Fairness.</strong> Three axes, only one measured against a standard definition.
  </p>
  <ul style="padding-left: 20px; margin-bottom: 20px;">
    <li style="margin-bottom: 8px;"><strong>Between datasets (the one that matters most).</strong> LNDb and LIDC IDRI differ in hospitals, scanners and readers. Trained on both, the model has a QWK of 0.557 in LNDb and 0.686 in LIDC IDRI on the 1 to 5 scale, and the gap grows on the binary scale: sensitivity of 0.50 in LNDb against 0.82 in LIDC IDRI, at a specificity of 0.90 and 0.85. A single threshold misses half of the suspicious LNDb nodules, which is why it must be set on local data (notebook 06).</li>
    <li style="margin-bottom: 8px;"><strong>Between nodules of different size.</strong> Small nodules are harder for every method, including the diameter-only baseline, because the same absolute error weighs more on a small nodule.</li>
    <li style="margin-bottom: 8px;"><strong>Between readers.</strong> The model agrees with each reader as well as one reader agrees with the others (QWK 0.533 and 0.555 against 0.527 and 0.528), so it is not systematically closer to any of them.</li>
  </ul>
  <p>
    <strong>What we could not check.</strong> LIDC IDRI does not expose sex, age or ethnicity in a form that survives anonymisation through <code>pylidc</code>, and LNDb does not either, so subgroup fairness by demographic is not measurable on the training data. A model of this kind cannot be deployed without a local fairness audit.
  </p>
  <p>
    <strong>Limitations.</strong> Interpretability is checked globally, not locally; fairness between datasets, sizes and readers, but not across demographic groups. 
  </p>
  <h1 style="color: #c25719; border-bottom: 2px solid #c25719; padding-bottom: 8px; margin-top: 0;">
    References
  </h1>
  <ul style="padding-left: 20px; margin-bottom: 20px;">
    <li style="margin-bottom: 8px;">Ambroise, C. and McLachlan, G. J. (2002). Selection bias in gene extraction on the basis of microarray gene expression data. <i>PNAS</i>, 99(10), 6562 to 6566.</li>
    <li style="margin-bottom: 8px;">Armato, S. G. III et al. (2011). The Lung Image Database Consortium (LIDC) and Image Database Resource Initiative (IDRI). <i>Medical Physics</i>, 38(2), 915 to 931.</li>
    <li style="margin-bottom: 8px;">Baltatzis, V. et al. (2021). The pitfalls of sample selection: a case study on lung nodule classification. arXiv:2108.05386.</li>
    <li style="margin-bottom: 8px;">Bankier, A. A. et al. (2017). Recommendations for measuring pulmonary nodules at CT: a statement from the Fleischner Society. <i>Radiology</i>, 285(2), 584 to 600.</li>
    <li style="margin-bottom: 8px;">Bergstra, J. and Bengio, Y. (2012). Random search for hyper-parameter optimization. <i>Journal of Machine Learning Research</i>, 13, 281 to 305.</li>
    <li style="margin-bottom: 8px;">Brodersen, K. H., Ong, C. S., Stephan, K. E. and Buhmann, J. M. (2010). The balanced accuracy and its posterior distribution. <i>Proceedings of the 20th International Conference on Pattern Recognition</i>, 3121 to 3124.</li>
    <li style="margin-bottom: 8px;">Carrazza, M., Kennedy, B., Rasin, A., Furst, J. and Raicu, D. (2016). Investigating the effects of majority voting on CAD systems: a LIDC case study. <i>Medical Imaging 2016: Computer Aided Diagnosis</i>, SPIE.</li>
    <li style="margin-bottom: 8px;">Causey, J. L., Zhang, J., Ma, S., Jiang, B., Qualls, J. A., Politte, D. G., Prior, F., Zhang, S., & Huang, X. (2018). <i>Highly accurate model for prediction of lung nodule malignancy with CT scans</i>. Scientific Reports, 8, 9286</li>
    <li style="margin-bottom: 8px;">Cawley, G. C. and Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. <i>JMLR</i>, 11, 2079 to 2107.</li>
    <li style="margin-bottom: 8px;">Cohen, J. (1988). <i>Statistical Power Analysis for the Behavioral Sciences</i> (2nd ed.). Lawrence Erlbaum Associates.</li>
    <li style="margin-bottom: 8px;">Cohen, J. (1968). <i>Weighted kappa: Nominal scale agreement with provision for scaled disagreement or partial credit</i>. Psychological Bulletin, 70(4), 213 to 220. https://doi.org/10.1037/h0026256</li>
    <li style="margin-bottom: 8px;">Collins, G. S. et al. (2024). TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. <i>BMJ</i>, 385, e078378.</li>
    <li style="margin-bottom: 8px;">Corral Fontecha et al. (2026). Physically aware radiomics without interpolation. arXiv:2607.12399.</li>
    <li style="margin-bottom: 8px;">Demircioğlu, A. (2021). Measuring the bias of incorrect application of feature selection when using cross validation in radiomics. <i>Insights into Imaging</i>, 12, 172.</li>
    <li style="margin-bottom: 8px;">Field, C. A. and Welsh, A. H. (2007). Bootstrapping clustered data. <i>Journal of the Royal Statistical Society B</i>, 69(3), 369 to 390.</li>
    <li style="margin-bottom: 8px;">Fluss, R., Faraggi, D. and Reiser, B. (2005). Estimation of the Youden index and its associated cutoff point. <i>Biometrical Journal</i>, 47(4), 458 to 472.</li>
    <li style="margin-bottom: 8px;">Frank, E. and Hall, M. (2001). A simple approach to ordinal classification. <i>Machine Learning: ECML 2001</i>, Lecture Notes in Computer Science 2167, 145 to 156. Springer.</li>
    <li style="margin-bottom: 8px;">Gneiting, T. (2011). Making and evaluating point forecasts. <i>Journal of the American Statistical Association</i>, 106(494), 746 to 762.</li>
    <li style="margin-bottom: 8px;">Grinsztajn, L., Oyallon, E. and Varoquaux, G. (2022). Why do tree based models still outperform deep learning on typical tabular data? <i>Advances in Neural Information Processing Systems 35</i> (NeurIPS 2022), Datasets and Benchmarks track.</li>
    <li style="margin-bottom: 8px;">Hanley, J. A. and McNeil, B. J. (1982). The meaning and use of the area under a receiver operating characteristic (ROC) curve. <i>Radiology</i>, 143(1), 29 to 36.</li>
    <li style="margin-bottom: 8px;">Hastie, T., Tibshirani, R. and Friedman, J. (2009). <i>The Elements of Statistical Learning</i> (2nd ed.), section 7.10. Springer.</li>
    <li style="margin-bottom: 8px;">He, H., & Garcia, E. A. (2009). <i>Learning from Imbalanced Data.</i> IEEE Transactions on Knowledge and Data Engineering, 21(9), 1263 to 1284. https://doi.org/10.1109/TKDE.2008.239</li>
    <li style="margin-bottom: 8px;">Hsu, C. W., Chang, C. C. and Lin, C. J. (2003). A practical guide to support vector classification. Technical report, Department of Computer Science, National Taiwan University.</li>
    <li style="margin-bottom: 8px;">Johnson, W. E., Li, C. and Rabinovic, A. (2007). Adjusting batch effects in microarray expression data using empirical Bayes methods. <i>Biostatistics</i>, 8(1), 118 to 127.</li>
    <li style="margin-bottom: 8px;">Kapoor, S. and Narayanan, A. (2023). Leakage and the reproducibility crisis in machine-learning-based science. <i>Patterns</i>, 4(9), 100804.</li>
    <li style="margin-bottom: 8px;">Koo, T. K. and Li, M. Y. (2016). A guideline of selecting and reporting intraclass correlation coefficients for reliability research. <i>Journal of Chiropractic Medicine</i>, 15(2), 155 to 163.</li>
    <li style="margin-bottom: 8px;">Krstajic, D., Buturovic, L. J., Leahy, D. E. and Thomas, S. (2014). Cross-validation pitfalls when selecting and assessing regression and classification models. <i>Journal of Cheminformatics</i>, 6, 10.</li>
    <li style="margin-bottom: 8px;">Lambin, P. et al. (2017). Radiomics: the bridge between medical imaging and personalized medicine. <i>Nature Reviews Clinical Oncology</i>, 14, 749 to 762.</li>
    <li style="margin-bottom: 8px;">Leeflang, M. M. G., Moons, K. G. M., Reitsma, J. B. and Zwinderman, A. H. (2008). Bias in sensitivity and specificity caused by data driven selection of optimal cutoff values: mechanisms, magnitude, and solutions. <i>Clinical Chemistry</i>, 54(4), 729 to 737.</li>
    <li style="margin-bottom: 8px;">Lei, Y., Zhu, H., Zhang, J., & Shan, H. (2020). <i>Meta Ordinal Regression Forest For Learning with Unsure Lung Nodules</i>. IEEE International Conference on Bioinformatics and Biomedicine (BIBM). https://arxiv.org/abs/2012.03480</li>
    <li style="margin-bottom: 8px;">Liu, L., Liu, Y. and Zhao, H. (2018). Benign and malignant solitary pulmonary nodules classification based on CNN and SVM. <i>ACM International Conference Proceeding Series</i>.</li>
    <li style="margin-bottom: 8px;">MacMahon, H. et al. (2017). Guidelines for management of incidental pulmonary nodules detected on CT images: from the Fleischner Society 2017. <i>Radiology</i>, 284(1), 228 to 243.</li>
    <li style="margin-bottom: 8px;">McCullagh, P. (1980). Regression models for ordinal data. <i>Journal of the Royal Statistical Society B</i>, 42(2), 109 to 142.</li>
    <li style="margin-bottom: 8px;">McNitt-Gray, M. F. et al. (2007). The Lung Image Database Consortium (LIDC) data collection process for nodule detection and annotation. <i>Academic Radiology</i>, 14(12), 1464 to 1474.</li>
    <li style="margin-bottom: 8px;">Molnar, C. (2022). Interpretable Machine Learning (2nd ed.). christophm.github.io/interpretable-ml-book.</li>
    <li style="margin-bottom: 8px;">Mongan, J., Moy, L. and Kahn, C. E. (2020). Checklist for Artificial Intelligence in Medical Imaging (CLAIM). <i>Radiology: Artificial Intelligence</i>, 2(2), e200029.</li>
    <li style="margin-bottom: 8px;">Orlhac, F. et al. (2022). A guide to ComBat harmonization of imaging biomarkers in multicenter studies. <i>Journal of Nuclear Medicine</i>, 63(2), 172 to 179.</li>
    <li style="margin-bottom: 8px;">Pai, S. et al. (2024). Foundation model for cancer imaging biomarkers. <i>Nature Machine Intelligence</i>.</li>
    <li style="margin-bottom: 8px;">Parmar, C., Grossmann, P., Bussink, J., Lambin, P. and Aerts, H. J. W. L. (2015). Machine learning methods for quantitative radiomic biomarkers. <i>Scientific Reports</i>, 5, 13087.</li>
    <li style="margin-bottom: 8px;">Pedrosa, J. et al. (2019). LNDb: a lung nodule database on computed tomography. arXiv:1911.08434.</li>
    <li style="margin-bottom: 8px;">Peoples et al. (2025). Finding reproducible and prognostic radiomic features in variable slice thickness contrast enhanced CT of colorectal liver metastases. <i>Journal of Machine Learning for Biomedical Imaging</i>, 2, 032.</li>
    <li style="margin-bottom: 8px;">Perkins, N. J. and Schisterman, E. F. (2006). The inconsistency of "optimal" cutpoints obtained using two criteria based on the receiver operating characteristic curve. <i>American Journal of Epidemiology</i>, 163(7), 670 to 675.</li>
    <li style="margin-bottom: 8px;">Probst, P. and Boulesteix, A. L. (2018). To tune or not to tune the number of trees in random forest. <i>Journal of Machine Learning Research</i>, 18(181), 1 to 18.</li>
    <li style="margin-bottom: 8px;">Probst, P., Wright, M. N. and Boulesteix, A. L. (2019). Hyperparameters and tuning strategies for random forest. <i>WIREs Data Mining and Knowledge Discovery</i>, 9(3), e1301.</li>
    <li style="margin-bottom: 8px;">PyRadiomics documentation, Customization (resampling and interpolators).</li>
    <li style="margin-bottom: 8px;">Regulation (EU) 2016/679 (GDPR); Regulation (EU) 2017/745 (MDR); Regulation (EU) 2024/1689 (AI Act).</li>
    <li style="margin-bottom: 8px;">Ross, J. C., Miller, J. V., Turner, W. D. and Kelliher, T. P. (2007). An analysis of early studies released by the Lung Imaging Database Consortium (LIDC). <i>Academic Radiology</i>, 14(11), 1382 to 1388.</li>
    <li style="margin-bottom: 8px;">Shafiq-ul-Hassan, M. et al. (2017). Intrinsic dependencies of CT radiomic features on voxel size and number of gray levels. <i>Medical Physics</i>, 44(3), 1050 to 1062.</li>
    <li style="margin-bottom: 8px;">Shen, W., Zhou, M., Yang, F., Yang, C. and Tian, J. (2015). Multi scale convolutional neural networks for lung nodule classification. <i>Information Processing in Medical Imaging</i>, Lecture Notes in Computer Science.</li>
    <li style="margin-bottom: 8px;">Shrout, P. E. and Fleiss, J. L. (1979). Intraclass correlations: uses in assessing rater reliability. <i>Psychological Bulletin</i>, 86(2), 420 to 428.</li>
    <li style="margin-bottom: 8px;">Steyerberg, E. W. and Harrell, F. E. (2016). Prediction models need appropriate internal, internal external, and external validation. <i>Journal of Clinical Epidemiology</i>, 69, 245 to 247.</li>
    <li style="margin-bottom: 8px;">Tejani, A. S. et al. (2024). Checklist for Artificial Intelligence in Medical Imaging (CLAIM): 2024 update. <i>Radiology: Artificial Intelligence</i>.</li>
    <li style="margin-bottom: 8px;">Traverso, A. et al. (2018). Repeatability and reproducibility of radiomic features: a systematic review. <i>International Journal of Radiation Oncology, Biology, Physics</i>, 102(4), 1143 to 1158.</li>
    <li style="margin-bottom: 8px;">Van Calster, B., McLernon, D. J., van Smeden, M., Wynants, L. and Steyerberg, E. W. (2019). Calibration: the Achilles heel of predictive analytics. <i>BMC Medicine</i>, 17, 230.</li>
    <li style="margin-bottom: 8px;">van Griethuysen, J. J. M. et al. (2017). Computational radiomics system to decode the radiographic phenotype. <i>Cancer Research</i>, 77(21), e104 to e107.</li>
    <li style="margin-bottom: 8px;">Varoquaux, G. (2018). Cross-validation failure: small sample sizes lead to large error bars. <i>NeuroImage</i>, 180, 68 to 77.</li>
    <li style="margin-bottom: 8px;">Yang, S., Wu, N., Zhang, L. and Li, M. (2021). Evaluation of the linear interpolation method in correcting the influence of slice thicknesses on radiomic feature values in solid pulmonary nodules: a prospective patient study. <i>Annals of Translational Medicine</i>, 9(4), 279.</li>
    <li style="margin-bottom: 8px;">Youden, W. J. (1950). Index for rating diagnostic tests. <i>Cancer</i>, 3(1), 32 to 35.</li>
    <li style="margin-bottom: 8px;">Zech, J. R. et al. (2018). Variable generalization performance of a deep learning model to detect pneumonia in chest radiographs: a cross sectional study. <i>PLoS Medicine</i>, 15(11), e1002683.</li>
    <li style="margin-bottom: 8px;">Zhang, H., Gu, X., Zhang, M., Yu, W., Chen, L., Wang, Z., Yao, F., Gu, Y., & Yang, G.-Z. (2022). <i>Re-thinking and Re-labeling LIDC-IDRI for Robust Pulmonary Cancer Prediction</i>. https://arxiv.org/abs/2207.14238</li>
    <li style="margin-bottom: 8px;">Zhang, Y., Jenkins, D. F., Manimaran, S. and Johnson, W. E. (2018). Alternative empirical Bayes models for adjusting for batch effects in genomic studies. <i>BMC Bioinformatics</i>, 19, 262.</li>
    <li style="margin-bottom: 8px;">Zou, H. and Hastie, T. (2005). Regularization and variable selection via the elastic net. <i>Journal of the Royal Statistical Society B</i>, 67(2), 301 to 320.</li>
    <li style="margin-bottom: 8px;">Zwanenburg, A. et al. (2019). Assessing robustness of radiomic features by image perturbation. <i>Scientific Reports</i>, 9, 614.</li>
    <li style="margin-bottom: 8px;">Zwanenburg, A. et al. (2020). The Image Biomarker Standardization Initiative: standardized quantitative radiomics for high-throughput image-based phenotyping. <i>Radiology</i>, 295(2), 328 to 338.</li>
</ul>
</div>